In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import utils

from dataset import load_uci_dataset, encode_labels, get_holdout_dataloaders
from models import MLP
from losses import PolyLoss, CombinedCLoss
from engine import train_model, evaluate_model
from plot_utils import plot_training_history
from gdv_utils import calculate_gdv

print("=== 1. SETUP AMBIENTE E DOWNLOAD DATI ===")

config_init = utils.carica_configurazione("config.yaml")
dataset_id = config_init['dataset']['uci_id']
device = torch.device(config_init['training']['device'] if torch.cuda.is_available() else "cpu")
print(f"Dispositivo in uso: {device}")

print(f"\nScaricamento dataset UCI ID {dataset_id}...")
X_raw, y_raw = load_uci_dataset(dataset_id)
y_encoded, n_classes = encode_labels(y_raw)
in_dim = X_raw.shape[1]

print("Calcolo GDV geometrico iniziale in corso...")
X_scaled_baseline = StandardScaler().fit_transform(X_raw)
X_tensor = torch.tensor(X_scaled_baseline, dtype=torch.float32)
y_tensor = torch.tensor(y_encoded, dtype=torch.long)

GDV_INIZIALE = calculate_gdv(X_tensor, y_tensor)

print(f"\n--- Dimensioni Rilevate dal Dataset (UCI {dataset_id}) ---")
print(f"Features (Input): {in_dim}")
print(f"Classi (Output):  {n_classes}")
print(f"GDV Dati Grezzi:  {GDV_INIZIALE:.4f}")
print("===================================================")

In [ ]:
print("=== 2. RICARICAMENTO CONFIGURAZIONE E DATALOADER ===")

config = utils.carica_configurazione("config.yaml")

# Imposta il seed per la riproducibilità degli split
random_state = config['dataset'].get('random_state', 42)
torch.manual_seed(random_state)

# --- CREAZIONE VELOCE DATALOADER ---
batch_size = config['dataset']['batch_size']
train_loader, val_loader, test_loader = get_holdout_dataloaders(
    X_raw, y_encoded, batch_size=batch_size, random_state=random_state
)

# --- SETUP ARCHITETTURA DINAMICA ---
config_layers = config['model'].get('hidden_layers', "auto")

if config_layers == "auto":
    # Calcola da solo i layer migliori in base a in_dim
    hidden_layers = utils.get_auto_hidden_layers(in_dim, n_classes)
    print(f"Architettura Auto-Selezionata: {in_dim} -> {hidden_layers} -> {n_classes}")
    # Sovrascrive la config in RAM per le celle di addestramento
    config['model']['hidden_layers'] = hidden_layers 
else:
    print(f"Architettura Forzata da YAML: {in_dim} -> {config_layers} -> {n_classes}")

In [ ]:
# === ESPERIMENTO 1: BASELINE (CROSS-ENTROPY) ===
print(f"\nAvvio Addestramento: Cross-Entropy su Dataset UCI ID {dataset_id} (Hold-Out)")

torch.manual_seed(config['dataset'].get('random_state', 42))

model_ce = MLP(
    input_size=in_dim, 
    num_classes=n_classes,
    hidden_layers=config['model']['hidden_layers'],
    dropout_rate=config['model']['dropout_rate']
).to(device)

criterion_ce = nn.CrossEntropyLoss()
optimizer_ce = optim.Adam(model_ce.parameters(), lr=config['training']['learning_rate'])

gdv_layer = model_ce.get_layer_for_gdv(index=config['model']['gdv_layer_index'])

history_ce, best_epoch_ce = train_model(
    model=model_ce, criterion=criterion_ce, optimizer=optimizer_ce, 
    train_loader=train_loader, val_loader=val_loader, device=device,
    num_epochs=config['training']['epochs'], layer_for_gdv=gdv_layer
)

test_acc_ce = evaluate_model(model_ce, test_loader, device)
history_ce['test_acc'] = test_acc_ce
print(f"Accuratezza Test Finale: {test_acc_ce:.2f}%")

# MLOps: Salvataggio log e Grafici
nome_exp = f"HoldOut_CE_UCI_{dataset_id}"
utils.salva_storico_json(history_ce, nome_exp, directory="logs")
plot_training_history(history_ce, f"Cross-Entropy (UCI {dataset_id})", GDV_INIZIALE, save_dir="plots")

In [ ]:
# === ESPERIMENTO 2: POLYLOSS ===
print(f"\nAvvio Addestramento: PolyLoss su Dataset UCI ID {dataset_id} (Hold-Out)")

torch.manual_seed(config['dataset'].get('random_state', 42))

model_combinedCLoss = MLP(
    input_size=in_dim, 
    num_classes=n_classes,
    hidden_layers=config['model']['hidden_layers'],
    dropout_rate=config['model']['dropout_rate']
).to(device)

criterion_combinedCLoss = PolyLoss(epsilon=2.0)
optimizer_combinedCLoss = optim.Adam(model_combinedCLoss.parameters(), lr=config['training']['learning_rate'])

gdv_layer = model_combinedCLoss.get_layer_for_gdv(index=config['model']['gdv_layer_index'])

history_combinedCLoss, best_epoch_combinedCLoss = train_model(
    model=model_combinedCLoss, criterion=criterion_combinedCLoss, optimizer=optimizer_combinedCLoss, 
    train_loader=train_loader, val_loader=val_loader, device=device,
    num_epochs=config['training']['epochs'], layer_for_gdv=gdv_layer
)

test_acc_poly = evaluate_model(model_combinedCLoss, test_loader, device)
history_combinedCLoss['test_acc'] = test_acc_poly
print(f"Accuratezza Test Finale: {test_acc_poly:.2f}%")

nome_exp = f"HoldOut_PolyLoss_UCI_{dataset_id}"
utils.salva_storico_json(history_combinedCLoss, nome_exp, directory="logs")
plot_training_history(history_combinedCLoss, f"PolyLoss (UCI {dataset_id})", GDV_INIZIALE, save_dir="plots")

In [ ]:
# === ESPERIMENTO 3: Combined C-Loss ===
print(f"\nAvvio Addestramento: Combined C-Loss su Dataset UCI ID {dataset_id} (Hold-Out)")

torch.manual_seed(config['dataset'].get('random_state', 42))

model_combinedCLoss = MLP(
    input_size=in_dim, 
    num_classes=n_classes,
    hidden_layers=config['model']['hidden_layers'],
    dropout_rate=config['model']['dropout_rate']
).to(device)

criterion_combinedCLoss = CombinedCLoss(sigma=0.5, gamma=0.5)
optimizer_combinedCLoss = optim.Adam(model_combinedCLoss.parameters(), lr=config['training']['learning_rate'])

gdv_layer = model_combinedCLoss.get_layer_for_gdv(index=config['model']['gdv_layer_index'])

history_combinedCLoss, best_epoch_combinedCLoss = train_model(
    model=model_combinedCLoss, criterion=criterion_combinedCLoss, optimizer=optimizer_combinedCLoss, 
    train_loader=train_loader, val_loader=val_loader, device=device,
    num_epochs=config['training']['epochs'], layer_for_gdv=gdv_layer
)

test_acc_combinedCLoss = evaluate_model(model_combinedCLoss, test_loader, device)
history_combinedCLoss['test_acc'] = test_acc_combinedCLoss
print(f"Accuratezza Test Finale: {test_acc_combinedCLoss:.2f}%")

nome_exp = f"HoldOut_CombinedCLoss_UCI_{dataset_id}"
utils.salva_storico_json(history_combinedCLoss, nome_exp, directory="logs")
plot_training_history(history_combinedCLoss, f"Combined C-Loss (UCI {dataset_id})", GDV_INIZIALE, save_dir="plots")

In [ ]:
import random
import os
from sklearn.model_selection import StratifiedKFold
from dataset import create_dataloader

# --- FUNZIONE SCUDO PER LA RIPRODUCIBILITÀ ---
def imposta_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# === ESPERIMENTO K-FOLD DINAMICO ===
# Modifica nella Cella 4 del notebook

def run_kfold_experiment(loss_fn, model_name, X, y, in_dim, n_classes, config):
    # Estrazione parametri specifici per il K-Fold dal config
    k_splits = config.get('kfold', {}).get('k_splits', 10)
    epochs_kfold = config.get('kfold', {}).get('epochs', 20)
    
    batch_size = config['dataset']['batch_size']
    base_seed = config['dataset'].get('random_state', 42)
    
    print(f"\n=== AVVIO {k_splits}-FOLD CV: {model_name} ===")
    print(f"Configurazione: {epochs_kfold} epoche per fold")
    
    imposta_seed(base_seed)
    skf = StratifiedKFold(n_splits=k_splits, shuffle=True, random_state=base_seed)
    
    fold_accs = []
    fold_gdvs = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n--- Fold {fold+1}/{k_splits} ---")
        
        seed_corrente = base_seed + fold
        imposta_seed(seed_corrente)
        
        X_tr, X_va = X[train_idx], X[val_idx]
        y_tr, y_va = y[train_idx], y[val_idx]
        
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_va_scaled = scaler.transform(X_va)
        
        # Uso di create_dataloader dal modulo dataset
        tr_loader = create_dataloader(X_tr_scaled, y_tr, batch_size=batch_size, shuffle=True)
        va_loader = create_dataloader(X_va_scaled, y_va, batch_size=batch_size, shuffle=False)
        
        # Inizializzazione DynamicMLP
        model = MLP(
            input_size=in_dim, 
            num_classes=n_classes,
            hidden_layers=config['model']['hidden_layers'],
            dropout_rate=config['model']['dropout_rate']
        ).to(device)
        
        optimizer = optim.Adam(model.parameters(), lr=config['training']['learning_rate'])
        gdv_layer = model.get_layer_for_gdv(index=config['model']['gdv_layer_index'])
        
        # Addestramento con le epoche specifiche del K-Fold
        history, best_ep = train_model(
            model=model, criterion=loss_fn, optimizer=optimizer,
            train_loader=tr_loader, val_loader=va_loader, device=device,
            num_epochs=epochs_kfold, # <--- Parametro dinamico dal config
            layer_for_gdv=gdv_layer
        )
        
        # Valutazione
        acc = evaluate_model(model, va_loader, device)
        fold_accs.append(acc)
        
        best_gdv = history['val_gdv'][best_ep - 1] if history['val_gdv'] else 0.0
        fold_gdvs.append(best_gdv)
        
    return fold_accs, fold_gdvs

print(f"\n=== ESECUZIONE 10-FOLD CV SU UCI ID {dataset_id} ===")

# Esecuzione per i 3 modelli nella Cella 4
k_val = config.get('kfold', {}).get('k_splits', 10)
print(f"\n=== ESECUZIONE {k_val}-FOLD CV SU UCI ID {dataset_id} ===")

# Utilizziamo X_raw e y_encoded calcolati nella Cella 1
acc_ce, gdv_ce = run_kfold_experiment(nn.CrossEntropyLoss(), "Cross-Entropy", X_raw, y_encoded, in_dim, n_classes, config)
acc_poly, gdv_poly = run_kfold_experiment(PolyLoss(epsilon=2.0), "PolyLoss", X_raw, y_encoded, in_dim, n_classes, config)
acc_closs, gdv_closs = run_kfold_experiment(CombinedCLoss(sigma=0.5, gamma=0.5), "Combined C-Loss", X_raw, y_encoded, in_dim, n_classes, config)

# Salvataggio MLOps
kfold_results = {
    "dataset_id": dataset_id,
    "folds": 10,
    "Cross-Entropy": {"acc": acc_ce, "gdv": gdv_ce},
    "PolyLoss": {"acc": acc_poly, "gdv": gdv_poly},
    "Combined C-Loss": {"acc": acc_closs, "gdv": gdv_closs}
}
utils.salva_storico_json(kfold_results, f"KFold_Results_UCI_{dataset_id}", directory="logs")

print("\n" + "="*50)
print("--- RIEPILOGO FINALE K-FOLD (Accuratezza Media) ---")
print(f"1. Cross-Entropy: {np.mean(acc_ce):.2f}% (± {np.std(acc_ce):.2f}%)")
print(f"2. PolyLoss:      {np.mean(acc_poly):.2f}% (± {np.std(acc_poly):.2f}%)")
print(f"3. C-Loss:        {np.mean(acc_closs):.2f}% (± {np.std(acc_closs):.2f}%)")

In [ ]:
import os
from datetime import datetime

fig, ax1 = plt.subplots(figsize=(5.5, 4.0))

modelli = ['Cross-Entropy', 'PolyLoss', 'Combined C-Loss']
medie_acc = [np.mean(acc_ce), np.mean(acc_poly), np.mean(acc_closs)]
std_acc = [np.std(acc_ce), np.std(acc_poly), np.std(acc_closs)]

colori = ['#004488', '#BB5500', '#117733']

barre = ax1.bar(modelli, medie_acc, yerr=std_acc, capsize=5, 
                color=colori, alpha=0.85, edgecolor='black', linewidth=1.0,
                error_kw={'elinewidth': 1.0, 'capthick': 1.0})

min_acc = min(medie_acc) - max(std_acc) - 1.5
max_acc = max(medie_acc) + max(std_acc) + 1.5
ax1.set_ylim(min_acc, max_acc)

ax1.set_ylabel('Accuratezza Media (%)')
ax1.set_title(f'Confronto Prestazioni 10-Fold Cross Validation\n(Dataset UCI ID: {dataset_id})')

for i, (barra, media) in enumerate(zip(barre, medie_acc)):
    altezza = barra.get_height()
    base_error_bar = altezza - std_acc[i]
    y_text = min_acc + (base_error_bar - min_acc) / 2
    
    ax1.text(barra.get_x() + barra.get_width()/2., y_text, 
             f'{media:.2f}%', ha='center', va='center', 
             color='white', fontweight='bold', fontsize=10)

plt.grid(axis='y', linestyle='--')
plt.tight_layout()

# Salvataggio MLOps Dinamico
save_dir = "plots"
os.makedirs(save_dir, exist_ok=True)
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
filename_pdf = os.path.join(save_dir, f"{timestamp}_KFold_UCI_{dataset_id}_Acc.pdf")

plt.savefig(filename_pdf, format='pdf', bbox_inches='tight')
print(f"[*] Grafico K-Fold salvato per la tesi: {filename_pdf}")

plt.show()